<a href="https://colab.research.google.com/github/PSS-Grp/pss-pub/blob/main/attendance_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 勤怠管理アプリ（Flask）を Colab で動かす

このノートブックは、GitHubリポジトリ `PSS-Grp/pss-pub` の `怠/attendance` にあった
Flask製の勤怠管理アプリを、**Google Colab上で動作確認できるように修正したもの**を
実行するためのものです。

## 今回の改修で直した主な不具合・懸念点

1. **`datetime.py` が標準ライブラリと名前衝突（致命的バグ）**
   attendanceフォルダ内に `datetime.py` という自作ファイルがあり、
   `import datetime` と書くたびに標準ライブラリではなくこのファイル自身が
   読み込まれてしまい、`AttributeError` でアプリ全体が起動できない状態でした。
   → 使われていない不要なファイルだったため削除しました。

2. **`login.py` にシェルコマンドが直接書かれていて構文エラー**
   `flask db init` などターミナルで打つはずのコマンドがPythonコードとして
   そのまま書かれており、実行・importすると構文エラーになる状態でした。
   → 実行しても安全なメモ用ファイルに直しました（DB作成は元々 `admin.py` の
   `db.create_all()` で行われており、このファイル自体は使われていません）。

3. **SECRET_KEY・パスワードのハードコード**
   `app.secret_key` や `SECRET_KEY` に固定文字列が、`auth.py` には実際の
   パスワードが平文でそのままコードに書かれ、公開リポジトリに残っていました。
   → 環境変数から読み込み、無ければ安全なランダム値を自動生成する方式に変更。
   `auth.py` は環境変数 or 入力プロンプトからパスワードを受け取る方式に変更しました。

4. **DBファイルパスがカレントディレクトリ依存**
   `sqlite:///db/attendance.db` という相対パスで、実行場所によっては
   DBファイルが見つからず失敗する状態でした。
   → スクリプト自身の場所を基準にした絶対パスに変更しました。

5. **新しいバージョンのFlask-SQLAlchemy / Flask-Adminとの非互換**
   最新版のライブラリでは `db.create_all()` にアプリケーションコンテキストが
   必須になっていたり、`Admin(..., template_mode='bootstrap4')` という書き方が
   廃止されていたため、現在のバージョンに合わせて修正しました。

6. **元のSQLiteデータベースに実データが入ったままコミットされていた**
   `db/attendance.db` に実際の従業員番号・氏名・パスワードハッシュ・勤怠記録が
   含まれていた可能性があるため、このノートブックには含めていません。
   代わりに `seed_demo_user.py` が、動作確認用のアカウントを作成します。

7. **データベース管理画面(/admin)に誰でもアクセスできてしまっていた**
   認証が一切無く、URLさえ知っていれば誰でもUser（従業員）・Time（勤怠記録）
   テーブルを閲覧・編集できる状態でした。
   → `User` テーブルに `is_admin` フラグを追加し、管理者専用アカウントで
   ログインした場合のみ `/admin` にアクセスできるようにしました
   （一般の従業員アカウントでログインしても管理画面には入れません。
   ログインすると自動的に管理画面へ遷移します）。

8. **会館名（勤務先）がユーザーによらず全員共通・固定の一覧だった**
   本葬・通夜の出勤画面にある「会館」のプルダウンは、`項目4`〜`項目8`のような
   仮の項目を含む全ユーザー共通の固定リストがHTMLに直接書かれていました。
   → 元々定義だけされて使われていなかった `Place` テーブルを活用し、
   ログインしているユーザーごとに異なる会館名の一覧を表示できるようにしました。
   「その他」は引き続き一覧の末尾に残し、選ぶと今まで通り会館名を手入力できます。
   会館名は管理者アカウントでログインし、`/admin/place/` から追加・編集できます
   （`user_id`を指定するとその従業員専用、空欄のままなら全員共通の会館になります）。

9. **管理画面に何も表示されない空の「Home」タブがあった**
   → ナビゲーションから非表示にし、管理者アカウントでログインした際の
   遷移先も、空のHome画面ではなくUser一覧(`/admin/user/`)に変更しました。

10. **Time（勤怠記録）をユーザーごとにCSVで出力する手段が無かった**
    → `/admin/time/` の一覧画面でCSVエクスポート機能を有効化し、
    従業員番号で検索・絞り込んでから出力すれば、その人の記録だけをCSVで
    ダウンロードできるようにしました（詳細は末尾の「補足」を参照）。

## 使い方

1. 下のセルを順番に実行してください。
2. 「ファイルを選択」が表示されたら、お渡しした `attendance_fixed.zip` を
   アップロードしてください。
3. 手順4のセルを実行すると、ノートブック内にアプリの画面が埋め込まれます。
   - 出退勤の打刻を試す場合 → 従業員番号: `0001` / パスワード: `demo1234`
     （会館名: 愛知葬祭 春日井会場、平安会館 一宮斎場）
   - 別の従業員として会館一覧の違いを見る場合 →
     `0002`/`demo2345`（名古屋メモリアルホール、豊田会館）、
     `0003`/`demo3456`（岡崎セレモニーホール、安城会館、刈谷会館）
   - データベース管理画面(/admin)を試す場合 → 従業員番号: `9001` / パスワード: `admin1234`
     （ログインすると自動的に管理画面へ移動します）


## 1. 修正済みファイル（zip）をアップロード

In [1]:
import os

if os.path.isdir("attendance_fixed"):
    print("attendance_fixed フォルダは既にあります。再アップロードは不要です。")
else:
    try:
        from google.colab import files
        print("attendance_fixed.zip を選択してアップロードしてください。")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
    except ImportError:
        # Colab以外（ローカルJupyter等）で実行する場合は、
        # あらかじめ同じフォルダに attendance_fixed.zip を置いておく
        zip_name = "attendance_fixed.zip"

    import zipfile
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(".")
    print("展開しました:", os.listdir("attendance_fixed"))


attendance_fixed.zip を選択してアップロードしてください。


Saving attendance_fixed.zip to attendance_fixed.zip
展開しました: ['honso.py', '__init__.py', 'index.py', 'requirements.txt', 'db', 'auth.py', 'tsuya.py', 'templates', 'admin.py', 'run_colab.py', 'login.py', 'static', 'models.py', 'seed_demo_user.py']


## 2. 依存ライブラリのインストール

In [2]:
!pip install -q -r attendance_fixed/requirements.txt
print("インストール完了")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 5.7 MB/s eta 0:00:00
インストール完了


## 3. データベースの初期化 & デモアカウント作成

前述の通り、実際の従業員データを含む元のDBは使わず、このノートブック用に
空のDBと、動作確認用のアカウント・会館名を新規作成します。

- 従業員アカウント（出退勤の打刻用。それぞれ会館の一覧が異なります）
  - `0001` / `demo1234` → 愛知葬祭 春日井会場、平安会館 一宮斎場
  - `0002` / `demo2345` → 名古屋メモリアルホール、豊田会館
  - `0003` / `demo3456` → 岡崎セレモニーホール、安城会館、刈谷会館
- 管理者アカウント（`/admin` のデータベース管理画面用）: 従業員番号 `9001` / パスワード `admin1234`

どのアカウントも同じログイン画面からログインしますが、管理画面には管理者
アカウントでログインした場合のみ入れます。会館名や従業員は `/admin/place/`・
`/admin/user/` から自由に追加・編集できます。

In [3]:
import sys, os

APP_DIR = os.path.abspath("attendance_fixed")
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

# db/attendance.db が残っていれば一度リセットして作り直す場合はコメントを外す
# os.remove(os.path.join(APP_DIR, "db", "attendance.db"))

os.chdir(APP_DIR)
import seed_demo_user
seed_demo_user.main()


デモ用アカウントを準備します。
  従業員アカウントを作成しました。 従業員番号: 0001 / パスワード: demo1234
    会館名を登録しました: 愛知葬祭 春日井会場, 平安会館 一宮斎場
  従業員アカウントを作成しました。 従業員番号: 0002 / パスワード: demo2345
    会館名を登録しました: 名古屋メモリアルホール, 豊田会館
  従業員アカウントを作成しました。 従業員番号: 0003 / パスワード: demo3456
    会館名を登録しました: 岡崎セレモニーホール, 安城会館, 刈谷会館
  管理者アカウントを作成しました。 従業員番号: 9001 / パスワード: admin1234


## 4. アプリを起動して画面を表示

Flaskサーバーをバックグラウンドスレッドで起動し、Colab内蔵の機能で
ノートブック上にアプリの画面を埋め込みます（ngrokなどの外部サービスは不要です）。

ログイン画面が表示されたら、出退勤の打刻を試す場合は従業員番号 `0001` / パスワード `demo1234` で、
管理画面(/admin)を試す場合は従業員番号 `9001` / パスワード `admin1234` でログインしてください。

In [9]:
import run_colab
run_colab.start(port=5000)


サーバーは既に起動しています（ポート 5000）。


<IPython.core.display.Javascript object>

## 5. データベース管理画面（Flask-Admin）を開く

`/admin/` にアクセスすると、User（従業員）・Time（勤怠記録）・Place（会館名）の
テーブルをブラウザから直接閲覧・編集できるFlask-Adminの管理画面が開きます。
何も表示されない空の「Home」タブは、実用上不要なためナビゲーションから
非表示にしています。

**手順4のiframe内で管理者アカウント（従業員番号 `9001` / パスワード
`admin1234`）でログインすると、ログイン後に自動的にUser一覧
(`/admin/user/`)へ遷移します。** そのため、通常はこの手順5のセルを
実行する必要はありません。

未ログイン、または一般の従業員アカウント（`0001`など）でログインした状態で
`/admin/` を開こうとすると、ログイン画面にリダイレクトされ管理画面には
入れません。

このセルは、手順4のiframeで一般の従業員アカウントとしてログインしたままの
状態から、別のiframeとして管理画面だけを改めて開きたい場合に使います
（サーバーは3.で起動済みのものをそのまま使うので、再起動の必要はありません。
ただし表示するiframe自体は、手順4で最後にログインしたアカウントのセッションを
共有します）。

In [8]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, path="/admin/user/", height=720)
except ImportError:
    print("Colab環境ではないため、ブラウザで http://127.0.0.1:5000/admin/user/ を開いてください。")


<IPython.core.display.Javascript object>

補足: どうしても通常のブラウザタブ（アドレスバーあり）で開きたい場合は、
上に表示されたiframe内を右クリックし、「このフレームを新しいタブで開く」
（ブラウザにより表記は異なります）を選ぶと、そのURLが新しいタブで開きます。
そのアドレスバーの末尾を `/admin/`（スラッシュを忘れずに）などに書き換えれば、
他のページにも移動できます。

## 補足

- 会館名を追加・変更したい場合は、管理者アカウント（`9001`）でログインして
  `/admin/place/` を開き、新しい行を追加してください。`user_id` にその従業員の
  ID（`/admin/user/` で確認できます）を入れるとその人専用の会館になり、
  `user_id` を空欄のままにすると全ユーザー共通の会館として一覧に表示されます。
- **Time（勤怠記録）をユーザーごとにCSVで出力したい場合**は、`/admin/time/`
  を開き、上部の検索欄に従業員番号（例: `0001`）を入力して絞り込んだ後、
  一覧右上の「Export」→「Export CSV」を押してください。絞り込んだ従業員だけの
  記録がCSVとしてダウンロードされます。何も絞り込まずにExportすれば、
  全ユーザー分がまとめてダウンロードされます。
- ログイン用のパスワードハッシュを新しく作りたい場合は、ターミナルの代わりに
  次のセルのようにして `auth.py` の関数を呼び出せます。
- 本番環境として使う場合は、`ATTENDANCE_SECRET_KEY` 環境変数を固定値で設定し、
  実データの取り扱いについて改めてセキュリティ面を確認してください。

In [6]:
import os
os.environ["AUTH_PASSWORD"] = "hujiko0-"
!python auth.py


pw_hash = scrypt:32768:8:1$1JZlOeQZnMkV4taT$c4de8d7e1324fd9d6ac8326d8ba3d14c55fc55d3ae07aed97790270a2bd50cd45df618b1fd5e8523081a0c7e3efd70c11dfdf4082f673c204db522941cf49a14
